# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kbhutto256/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane: Refresh / Content Opportunity Scoring**

This is primarily a **classification** task, with a **ranking** step layered on top for the final output. The model classifies whether a page is a refresh candidate (binary: review / don't review) and produces a probability per page. Those probabilities are then used to **rank** pages into a queue, because the real decision isn't "yes or no" for every single page — it's "which N pages does a reviewer look at first, given limited weekly capacity." Classification supplies the signal; ranking turns that signal into something a human can actually act on.

**The action the output supports:** a content/SEO reviewer opens the top N pages from the ranked queue and decides whether to refresh, expand, consolidate, or leave each one. The model doesn't make the call — it decides who gets looked at first.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

task_type = "classification (probability per page) -> ranking (Precision@K review queue)"
print("ML task type:", task_type)

ML task type: classification (probability per page) -> ranking (Precision@K review queue)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Starter target (proxy):** `is_declining_label = trend_direction == "down"`. This is a current-window bucket calculated from the same window as the features, not an observed future outcome — it tells you a page's trend right now, not what will actually happen to it next. It's a reasonable teaching proxy, but a beginner one.

**Stronger label (for later, on the warehouse data):** a genuine future observed outcome — e.g. features from the prior 90 days used to predict decline or recovery over the *next* 30 days. This avoids the proxy problem because it measures something that actually happened after the feature window, rather than applying a rule to the same window the features come from.

In [2]:
import requests

# Define the URL for the raw CSV data
url = "https://raw.githubusercontent.com/kbhutto256/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

# Define the local path where the file should be saved
local_path = "data/raw/content_refresh_anonymized.csv"

# Create the directory if it doesn't exist
import os
os.makedirs(os.path.dirname(local_path), exist_ok=True)

# Download the file
response = requests.get(url)
response.raise_for_status() # Raise an exception for bad status codes

with open(local_path, 'wb') as f:
    f.write(response.content)

print(f"File downloaded successfully to {local_path}")

File downloaded successfully to data/raw/content_refresh_anonymized.csv


In [3]:
import pandas as pd
import os
import requests # Ensure requests is imported

local_path = 'data/raw/content_refresh_anonymized.csv'

# --- START: Ensure file is present ---
# Check if the file already exists to avoid unnecessary re-downloads
if not os.path.exists(local_path):
    print(f"File '{local_path}' not found, attempting to download.")

    # Define the URL for the raw CSV data (using the global variable if available, or define here)
    url = "https://raw.githubusercontent.com/kbhutto256/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

    # Create the directory if it doesn't exist
    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    try:
        # Download the file
        response = requests.get(url)
        response.raise_for_status() # Raise an exception for bad status codes

        with open(local_path, 'wb') as f:
            f.write(response.content)

        print(f"File downloaded successfully to {local_path}")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading file: {e}")
        raise # Re-raise the exception if download fails
    except Exception as e:
        print(f"An unexpected error occurred during file download: {e}")
        raise
else:
    print(f"File '{local_path}' already exists, skipping download.")
# --- END: Ensure file is present ---

# Now, proceed with reading the CSV, which should be guaranteed to exist
df = pd.read_csv(local_path)

print(df['trend_direction'].value_counts())
print(f"\n% of pages labeled 'down' (proxy target): {(df['trend_direction']=='down').mean():.1%}")

File 'data/raw/content_refresh_anonymized.csv' already exists, skipping download.
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

% of pages labeled 'down' (proxy target): 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50.** A reviewer can realistically act on roughly 50 flagged pages per review cycle, so this metric matches real decision capacity directly — "of the top 50 pages I flag, how many actually deserved review?" Plain accuracy would be misleading here since most pages are *not* declining, so a model could score high accuracy while catching almost nothing useful. ROC AUC is also weaker for this use case because it scores the whole ranking equally, when in practice only the very top of the queue is ever acted on. The starter pipeline already gives numbers to beat: baseline rule scored 0.240 Precision@50, the random forest scored 0.740.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Reference numbers verified from outputs/model_results.json and outputs/model_report.md
baseline_precision_at_50 = 0.240
random_forest_precision_at_50 = 0.740

print(f"Baseline rule Precision@50: {baseline_precision_at_50}")
print(f"Random forest Precision@50: {random_forest_precision_at_50}")
print(f"Gap: {random_forest_precision_at_50 - baseline_precision_at_50:.3f}")

Baseline rule Precision@50: 0.24
Random forest Precision@50: 0.74
Gap: 0.500


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page (`content_id`)**, after the starter pipeline's standard filters: `impressions_90d > 0` and `content_age_days >= 90`, deduplicated by `content_id`. This is page-level grain — not client-level, not query-level — because the decision this project supports ("which page should a reviewer look at first") is made one page at a time.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df_filtered = (
    df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)]
    .drop_duplicates(subset='content_id')
)

print(f"Rows before filtering: {len(df)}")
print(f"Rows after filtering (one row = one content page): {len(df_filtered)}")

cols_to_show = ['content_id', 'client_id', 'impressions_90d', 'content_age_days', 'trend_direction']
df_filtered[cols_to_show].head(10)

Rows before filtering: 30000
Rows after filtering (one row = one content page): 30000


,content_id,client_id,impressions_90d,content_age_days,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,187,down
1,content_a1fb4e703a9e,client_4e07408562,15320,445,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,141,down
3,content_331d6c4de07b,client_19581e27de,11751,463,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,263,down
5,content_d4084a4bc775,client_f369cb89fc,3970,147,down
6,content_9a34b442b552,client_8722616204,20,90,down
7,content_a63219c6e95a,client_19581e27de,1724,445,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,90,down
9,content_c27558df2b0c,client_19581e27de,1240,257,down


In [6]:
# Sketch of the target column itself

df_filtered = df_filtered.copy()
df_filtered['is_declining_label'] = (df_filtered['trend_direction'] == 'down').astype(int)

print(df_filtered['is_declining_label'].value_counts())
df_filtered[['content_id', 'trend_direction', 'is_declining_label']].head(10)

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


,content_id,trend_direction,is_declining_label
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (like the starter baseline) can only combine a handful of hand-picked thresholds in a fixed way — e.g. "flag if impressions >= 500 AND days-since-update >= 180." That misses interactions between signals: a page with only moderate impressions but a fast-declining trend *and* weak engagement might be a much bigger opportunity than a page that simply crosses one static threshold. The right combination of signals also differs by content type, position tier, and age — exactly the kind of non-linear interaction a tree-based model can learn from data but a human can't hand-write into an if-statement.

This isn't a guess — the starter pipeline already measured the gap directly: the hand-written rule reached Precision@50 of 0.240, while the random forest reached 0.740 on the same data. That's roughly three times as many true candidates caught in the top 50, using the exact same underlying signals — the only difference is that the model found interactions the rule couldn't express.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

rule_hits_per_50 = 0.240 * 50
model_hits_per_50 = 0.740 * 50

print(f"Fixed rule catches ~{rule_hits_per_50:.0f} true candidates in its top 50")
print(f"Learned model catches ~{model_hits_per_50:.0f} true candidates in its top 50")
print(f"That's roughly {model_hits_per_50/rule_hits_per_50:.1f}x more true candidates caught")

Fixed rule catches ~12 true candidates in its top 50
Learned model catches ~37 true candidates in its top 50
That's roughly 3.1x more true candidates caught


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.